# Riffs

This notebook contains standalone riff visualizations I created for the final project.

## Riff 1: Constitutional Age vs Document Length


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT_DIR = Path("iframe_figures")
OUT_DIR.mkdir(exist_ok=True)

lib = pd.read_csv("output/constitutions-LIB.csv", sep="|")
lib["original_year"] = pd.to_numeric(lib["original_year"], errors="coerce")
lib["revision_year"] = pd.to_numeric(lib["revision_year"], errors="coerce")
lib["n_chars"] = pd.to_numeric(lib["n_chars"], errors="coerce")
lib["n_provisions"] = pd.to_numeric(lib["n_provisions"], errors="coerce")

riff1 = lib.dropna(subset=["n_chars", "n_provisions"]).copy()
riff1 = riff1[(riff1["n_chars"] > 0) & (riff1["n_provisions"] > 0)]
riff1 = riff1.dropna(subset=["original_year"]).copy()
riff1["n_chars_k"] = riff1["n_chars"] / 1000
riff1["was_revised"] = riff1["revision_year"].notna()

top_longest = riff1.nlargest(6, "n_chars")
riff1[["country", "original_year", "n_chars", "n_provisions"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    riff1["original_year"],
    riff1["n_chars_k"],
    c=riff1["n_provisions"],
    cmap="plasma",
    s=np.where(riff1["was_revised"], 58, 34),
    alpha=0.78,
    linewidth=0.4,
    edgecolor="white",
)

for _, row in top_longest.iterrows():
    ax.annotate(
        row["country"],
        (row["original_year"], row["n_chars_k"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8,
        color="#222222",
    )

ax.set_xlabel("Original constitution year")
ax.set_ylabel("Constitution length, thousands of characters")
ax.set_title("Original Year and Constitution Length")
ax.grid(True, color="#dddddd", linewidth=0.7)

cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label("Number of provisions")

ax.scatter([], [], s=34, color="#7b3294", edgecolor="white", linewidth=0.4, label="No revision year")
ax.scatter([], [], s=58, color="#7b3294", edgecolor="white", linewidth=0.4, label="Has revision year")
ax.legend(frameon=False, loc="upper right")

fig.tight_layout()
fig_path = OUT_DIR / "riff1_original_year_vs_length.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
fig_path

## V-Dem Join

In [ ]:
import re
import unicodedata

VDEM_PATH = Path("sentiment_lexicon/V-Dem-CY-Core-v16.csv")

vdem_cols = [
    "country_name",
    "year",
    "v2x_polyarchy",
    "v2x_libdem",
    "v2x_partipdem",
    "v2x_delibdem",
    "v2x_egaldem",
]

dem_labels = {
    "v2x_polyarchy": "Electoral democracy",
    "v2x_libdem": "Liberal democracy",
    "v2x_partipdem": "Participatory democracy",
    "v2x_delibdem": "Deliberative democracy",
    "v2x_egaldem": "Egalitarian democracy",
}

name_map = {
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "Congo": "Republic of the Congo",
    "Cote DIvoire": "Ivory Coast",
    "Czech Republic": "Czechia",
    "East Timor": "Timor-Leste",
    "Gambia": "The Gambia",
    "German Federal Republic": "Germany",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Macedonia": "North Macedonia",
    "Micronesia": "Federated States of Micronesia",
    "Myanmar": "Burma/Myanmar",
    "Peoples Republic of Korea": "North Korea",
    "Republic of Korea": "South Korea",
    "Socialist Republic of Vietnam": "Vietnam",
    "St Kitts and Nevis": "St. Kitts and Nevis",
    "St Lucia": "St. Lucia",
    "St Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Surinam": "Suriname",
    "Swaziland": "Eswatini",
    "Turkey": "Turkiye",
}

def country_key(value):
    ascii_value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", ascii_value.lower()).strip()

vdem = pd.read_csv(VDEM_PATH, usecols=vdem_cols)
vdem["country_key"] = vdem["country_name"].map(country_key)

lib_vdem = lib.copy()
lib_vdem["vdem_country"] = lib_vdem["country"].replace(name_map)
lib_vdem["country_key"] = lib_vdem["vdem_country"].map(country_key)
lib_vdem["text_year"] = pd.to_numeric(lib_vdem["file_year"], errors="coerce").astype("Int64")
lib_vdem["chars_per_provision"] = lib_vdem["n_chars"] / lib_vdem["n_provisions"]

joined = lib_vdem.merge(
    vdem,
    left_on=["country_key", "text_year"],
    right_on=["country_key", "year"],
    how="left",
)

joined = joined.dropna(subset=["v2x_polyarchy", "n_chars", "n_provisions"]).copy()
joined["n_chars_k"] = joined["n_chars"] / 1000
joined["democracy_group"] = pd.cut(
    joined["v2x_polyarchy"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=["Lower", "Middle", "Higher"],
    include_lowest=True,
)

print(f"Matched {len(joined)} of {len(lib)} constitutions to V-Dem country-year observations.")
joined[["country", "text_year", "country_name", "v2x_polyarchy", "n_chars", "n_provisions"]].head()

## Riff 2: Electoral Democracy and Constitution Length

This riff looks at if more democratic country-years tend to be associated with longer constitutional texts. 

In [ ]:
riff2 = joined.dropna(subset=["v2x_polyarchy", "n_chars", "n_provisions"]).copy()
riff2_outliers = riff2.nlargest(6, "n_chars")

fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    riff2["v2x_polyarchy"],
    riff2["n_chars_k"],
    c=riff2["n_provisions"],
    cmap="viridis",
    s=42,
    alpha=0.78,
    linewidth=0.4,
    edgecolor="white",
)

for _, row in riff2_outliers.iterrows():
    ax.annotate(
        row["country"],
        (row["v2x_polyarchy"], row["n_chars_k"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8,
        color="#222222",
    )

ax.set_xlabel("V-Dem electoral democracy index")
ax.set_ylabel("Constitution length, thousands of characters")
ax.set_title("Electoral Democracy and Constitution Length")
ax.grid(True, color="#dddddd", linewidth=0.7)

cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label("Number of provisions")
fig.tight_layout()
fig_path = OUT_DIR / "riff2_vdem_polyarchy_vs_length.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
fig_path

## Riff 3: Democracy Indices and Structural Features


In [ ]:
riff3 = joined.dropna(subset=["democracy_group", "n_chars_k", "n_provisions"]).copy()
group_summary = riff3.groupby("democracy_group", observed=False).agg(
    constitutions=("constitution_id", "count"),
    average_length_k=("n_chars_k", "mean"),
    average_provisions=("n_provisions", "mean"),
).reset_index()

x = np.arange(len(group_summary))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.8), sharex=True)

axes[0].bar(x, group_summary["average_length_k"], color="#4c78a8")
axes[0].set_title("Average Length")
axes[0].set_ylabel("Thousands of characters")

axes[1].bar(x, group_summary["average_provisions"], color="#f58518")
axes[1].set_title("Average Provisions")
axes[1].set_ylabel("Number of provisions")

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(group_summary["democracy_group"])
    ax.set_xlabel("Electoral democracy group")
    ax.grid(True, axis="y", color="#dddddd", linewidth=0.7)
    for idx, row in group_summary.iterrows():
        ax.text(
            idx,
            ax.get_ylim()[1] * 0.03,
            f"n={int(row['constitutions'])}",
            ha="center",
            va="bottom",
            fontsize=9,
            color="#222222",
        )

fig.suptitle("Average Constitution Structure by Democracy Group", y=1.02)

fig.tight_layout()
fig_path = OUT_DIR / "riff3_vdem_grouped_structure.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
group_summary